# Experiment A: Bayesian Hierarchical Poisson with ELO Hyperpriors

**Hypothesis**: Per-team attack/defence parameters overfit severely on sparse LOTO folds (DC MLE: 3.76 pts vs GLM: 4.34 pts). A Bayesian hierarchical model with population-level hyperpriors anchored to ELO ratings should regularise team-specific parameters via shrinkage, fixing the overfitting without losing the team-level signal.

**Approach**: Baio & Blangiardo (2010) hierarchical Poisson, extended with ELO-informed hyperpriors on attack parameters.

**Baseline to beat**: Poisson GLM = 4.336 pts/match [3.930, 4.742]. Autofill = 4.137 pts/match.

In [1]:
import sys
sys.path.insert(0, '../../src')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import pymc as pm
import pytensor.tensor as pt
import arviz as az
import mlflow
from functools import lru_cache
from scipy.stats import poisson
from pathlib import Path

from evaluation.sporza import score_breakdown, sporza_points_series

DATA = Path('../../data/processed')
WC_YEARS = [2010, 2014, 2018, 2022]
MAX_GOALS = 8
RANDOM_SEED = 42

print(f'PyMC version: {pm.__version__}')
print(f'ArviZ version: {az.__version__}')

PyMC version: 6.0.1
ArviZ version: 1.2.0


In [2]:
# ── Sporza score maximiser (same as GLM notebook) ──────────────────────────
@lru_cache(maxsize=50000)
def best_score_cached(lh_r: float, la_r: float) -> tuple:
    """Return (pred_home, pred_away) that maximises expected Sporza pts."""
    goals = np.arange(MAX_GOALS + 1)
    ph_pmf = poisson.pmf(goals, lh_r)
    pa_pmf = poisson.pmf(goals, la_r)
    joint = np.outer(ph_pmf, pa_pmf)

    def sporza_mat(pred_h: int, pred_a: int) -> float:
        pts = 0.0
        for ah in range(MAX_GOALS + 1):
            for aa in range(MAX_GOALS + 1):
                p = joint[ah, aa]
                if p < 1e-9:
                    continue
                if pred_h == ah and pred_a == aa:
                    pts += p * 10
                elif (pred_h - pred_a) == (ah - aa):
                    pts += p * 7
                elif np.sign(pred_h - pred_a) == np.sign(ah - aa):
                    pts += p * 5
                else:
                    pts += p * 1
        return pts

    best_pts, best_h, best_a = -1.0, 1, 0
    for ph_idx in range(MAX_GOALS + 1):
        for pa_idx in range(MAX_GOALS + 1):
            ep = sporza_mat(ph_idx, pa_idx)
            if ep > best_pts:
                best_pts, best_h, best_a = ep, ph_idx, pa_idx
    return best_h, best_a


def bootstrap_ci(pts: pd.Series, n_boot: int = 10000) -> dict:
    rng = np.random.default_rng(RANDOM_SEED)
    vals = pts.values
    boot = np.array([rng.choice(vals, size=len(vals), replace=True).mean() for _ in range(n_boot)])
    lo, hi = np.percentile(boot, [2.5, 97.5])
    return {'mean': float(vals.mean()), 'ci_lo': float(lo), 'ci_hi': float(hi)}

In [3]:
# ── Build Bayesian model and get posterior predictive λs ───────────────────
def run_bayesian_fold(
    train: pd.DataFrame,
    eval_df: pd.DataFrame,
    year: int,
    draws: int = 1000,
    tune: int = 1000,
) -> tuple[np.ndarray, np.ndarray, dict]:
    """
    Fit a Bayesian hierarchical Poisson model on `train` and return
    posterior-mean (lambda_home, lambda_away) for each row in `eval_df`.

    Model (Baio & Blangiardo 2010 + ELO hyperpriors):
      log(λ_home) = intercept + attack[home] - defence[away] + home_adv
      log(λ_away) = intercept + attack[away] - defence[home]

      attack[i] ~ Normal(mu_attack[i], sigma_attack)   -- ELO informs mu
      defence[i] ~ Normal(0, sigma_defence)
    """
    # ── union of teams across train + eval ──────────────────────────────
    all_teams = sorted(set(train['home_team'].tolist() + train['away_team'].tolist() +
                           eval_df['home_team'].tolist() + eval_df['away_team'].tolist()))
    team2idx = {t: i for i, t in enumerate(all_teams)}
    n_teams = len(all_teams)

    def idx(col: pd.Series) -> np.ndarray:
        return col.map(team2idx).fillna(0).astype(int).values

    h_idx = idx(train['home_team'])
    a_idx = idx(train['away_team'])
    y_home = train['home_score'].values.astype(int)
    y_away = train['away_score'].values.astype(int)

    # ── ELO-based prior mean for attack ─────────────────────────────────
    # For each team, collect all non-null ELO observations and take the last one.
    elo_map: dict[str, float] = {}
    for _, row in train.sort_values('date').iterrows():
        if pd.notna(row['home_elo']) and row['home_elo'] > 0:
            elo_map[row['home_team']] = float(row['home_elo'])
        if pd.notna(row['away_elo']) and row['away_elo'] > 0:
            elo_map[row['away_team']] = float(row['away_elo'])

    # Also pull from eval teams that appear in train as away/home
    for _, row in eval_df.iterrows():
        if row['home_team'] not in elo_map and pd.notna(row.get('home_elo', np.nan)):
            elo_map[row['home_team']] = float(row['home_elo'])
        if row['away_team'] not in elo_map and pd.notna(row.get('away_elo', np.nan)):
            elo_map[row['away_team']] = float(row['away_elo'])

    valid_elos = [v for v in elo_map.values() if np.isfinite(v)]
    median_elo = float(np.median(valid_elos)) if valid_elos else 1500.0

    raw_elos = np.array([elo_map.get(t, median_elo) for t in all_teams], dtype=float)
    # Fill any remaining NaN/inf with median
    raw_elos = np.where(np.isfinite(raw_elos), raw_elos, median_elo)
    # Scale ELO to attack-parameter space: centre at 0, ±1 ≈ ±300 ELO pts
    elo_std = raw_elos.std()
    elo_scaled = ((raw_elos - raw_elos.mean()) / (elo_std + 1e-6) * 0.3).astype(float)

    assert np.all(np.isfinite(elo_scaled)), f"NaN in elo_scaled: {np.sum(~np.isfinite(elo_scaled))} bad values"

    # ── neutral-site array ───────────────────────────────────────────────
    is_neutral_tr = train['is_neutral'].fillna(0).values.astype(float)

    print(f'  Fold {year}: {len(train):,} train rows | {n_teams} teams | '
          f'{len(eval_df)} eval rows | ELO range [{raw_elos.min():.0f}, {raw_elos.max():.0f}]')

    coords = {'team': all_teams}
    with pm.Model(coords=coords) as model:  # noqa: F841
        # ── Priors ──────────────────────────────────────────────────────
        sigma_attack = pm.HalfNormal('sigma_attack', sigma=0.3)
        sigma_defence = pm.HalfNormal('sigma_defence', sigma=0.3)

        # Attack: ELO-informed mean per team; shared variance
        attack = pm.Normal('attack', mu=elo_scaled, sigma=sigma_attack, dims='team')
        # Defence: shrink toward 0 (lower = concede fewer goals)
        defence = pm.Normal('defence', mu=0.0, sigma=sigma_defence, dims='team')

        intercept = pm.Normal('intercept', mu=0.0, sigma=0.5)
        home_adv = pm.Normal('home_adv', mu=0.2, sigma=0.15)

        # ── Likelihood ──────────────────────────────────────────────────
        # Home advantage suppressed for neutral venues
        ha_eff = home_adv * (1.0 - is_neutral_tr)

        log_lh = intercept + attack[h_idx] - defence[a_idx] + ha_eff
        log_la = intercept + attack[a_idx] - defence[h_idx]

        lh_obs = pm.math.exp(log_lh)
        la_obs = pm.math.exp(log_la)

        _ = pm.Poisson('goals_home', mu=lh_obs, observed=y_home)
        _ = pm.Poisson('goals_away', mu=la_obs, observed=y_away)

        # ── Sampling ────────────────────────────────────────────────────
        idata = pm.sample(
            draws=draws,
            tune=tune,
            chains=2,
            random_seed=RANDOM_SEED,
            progressbar=True,
            target_accept=0.9,
        )

    # ── Posterior predictive λ for eval rows ────────────────────────────
    # Take posterior mean of attack/defence/intercept/home_adv
    post = idata.posterior
    atk_mean = post['attack'].values.reshape(-1, n_teams).mean(axis=0)
    def_mean = post['defence'].values.reshape(-1, n_teams).mean(axis=0)
    int_mean = float(post['intercept'].values.mean())
    ha_mean  = float(post['home_adv'].values.mean())

    eh_idx = idx(eval_df['home_team'])
    ea_idx = idx(eval_df['away_team'])
    is_neutral_ev = eval_df['is_neutral'].fillna(0).values.astype(float)

    lh_pred = np.exp(int_mean + atk_mean[eh_idx] - def_mean[ea_idx] + ha_mean * (1 - is_neutral_ev))
    la_pred = np.exp(int_mean + atk_mean[ea_idx] - def_mean[eh_idx])

    # Convergence diagnostics
    rhat = az.rhat(idata)
    max_rhat = float(max(
        rhat['attack'].values.max(),
        rhat['defence'].values.max(),
        float(rhat['intercept'].values.max()),
        float(rhat['home_adv'].values.max()),
    ))
    divergences = int(idata.sample_stats['diverging'].values.sum())
    diag = {
        'max_rhat': max_rhat,
        'divergences': divergences,
        'intercept_mean': int_mean,
        'home_adv_mean': ha_mean,
        'sigma_attack_mean': float(post['sigma_attack'].values.mean()),
        'sigma_defence_mean': float(post['sigma_defence'].values.mean()),
    }
    print(f'  → max R̂={max_rhat:.3f}  divergences={divergences}  '
          f'intercept={int_mean:.3f}  home_adv={ha_mean:.3f}')
    return lh_pred, la_pred, diag

In [4]:
import json
manifest = json.loads((DATA / 'split_manifest.json').read_text())
autofill_mean = manifest['autofill_baseline']['pooled_mean_pts']
print(f'Autofill baseline: {autofill_mean:.3f} pts/match')
print(f'GLM baseline:      4.336 pts/match  [3.930, 4.742]')

Autofill baseline: 4.137 pts/match
GLM baseline:      4.336 pts/match  [3.930, 4.742]


In [5]:
# ── LOTO-CV main loop ─────────────────────────────────────────────────────
all_pts: list[float] = []
fold_results: list[dict] = []
fold_diags: list[dict] = []

for year in WC_YEARS:
    print(f'\n=== Fold: WC {year} ===')
    train = pd.read_parquet(DATA / f'train_fold_{year}.parquet')
    evalf = pd.read_parquet(DATA / f'eval_fold_{year}.parquet')

    lh_pred, la_pred, diag = run_bayesian_fold(train, evalf, year)
    diag['year'] = year
    fold_diags.append(diag)

    preds = [
        best_score_cached(round(float(h), 2), round(float(a), 2))
        for h, a in zip(lh_pred, la_pred)
    ]
    pred_home = pd.Series([p[0] for p in preds])
    pred_away = pd.Series([p[1] for p in preds])

    bd = score_breakdown(
        pred_home, pred_away,
        evalf['home_score'].reset_index(drop=True),
        evalf['away_score'].reset_index(drop=True),
    )
    pts = sporza_points_series(
        pred_home, pred_away,
        evalf['home_score'].reset_index(drop=True),
        evalf['away_score'].reset_index(drop=True),
    )
    all_pts.extend(pts.tolist())
    fold_results.append({
        'year': year,
        'mean_pts': bd['mean_pts'],
        'pct_exact': bd['pct_exact'],
        'pct_correct_result': bd['pct_correct_result'],
    })
    print(f'  Sporza: {bd["mean_pts"]:.3f} pts  exact={bd["pct_exact"]:.1%}  correct={bd["pct_correct_result"]:.1%}')

Initializing NUTS using jitter+adapt_diag...



=== Fold: WC 2010 ===
  Fold 2010: 219 train rows | 133 teams | 64 eval rows | ELO range [906, 2098]


Multiprocess sampling (2 chains in 2 jobs)


NUTS: [sigma_attack, sigma_defence, attack, defence, intercept, home_adv]


Output()

...

Sampling 2 chains for 1_000 tune and 1_000 draw iterations (2_000 + 2_000 draws total) took 13 seconds.


We recommend running at least 4 chains for robust computation of convergence diagnostics


The rhat statistic is larger than 1.01 for some parameters. This indicates problems during sampling. See https://arxiv.org/abs/1903.08008 for details


The effective sample size per chain is smaller than 100 for some parameters.  A higher number is needed for reliable rhat and ess computation. See https://arxiv.org/abs/1903.08008 for details


  → max R̂=1.010  divergences=0  intercept=0.106  home_adv=0.290
  Sporza: 3.719 pts  exact=6.2%  correct=54.7%


In [6]:
pooled_pts = pd.Series(all_pts)
ci = bootstrap_ci(pooled_pts)

print('\n' + '='*60)
print('POOLED LOTO-CV RESULTS')
print('='*60)
print(f'Bayesian hierarchical Poisson: {ci["mean"]:.3f} pts/match')
print(f'  95% CI: [{ci["ci_lo"]:.3f}, {ci["ci_hi"]:.3f}]  (width={ci["ci_hi"]-ci["ci_lo"]:.3f})')
print()
print(f'Comparison:')
print(f'  vs Autofill (4.137): {ci["mean"] - autofill_mean:+.3f} pts')
print(f'  vs GLM      (4.336): {ci["mean"] - 4.336:+.3f} pts')
print()
print('Per-fold:')
for r in fold_results:
    print(f'  WC {r["year"]}: {r["mean_pts"]:.3f} pts  exact={r["pct_exact"]:.1%}  correct={r["pct_correct_result"]:.1%}')

beats_autofill = ci['ci_lo'] > autofill_mean
beats_glm = ci['ci_lo'] > 4.336
print(f'\nCI lower > autofill: {beats_autofill}')
print(f'CI lower > GLM:      {beats_glm}')


POOLED LOTO-CV RESULTS
Bayesian hierarchical Poisson: 4.270 pts/match
  95% CI: [3.887, 4.660]  (width=0.773)

Comparison:
  vs Autofill (4.137): +0.133 pts
  vs GLM      (4.336): -0.066 pts

Per-fold:
  WC 2010: 4.391 pts  exact=14.1%  correct=56.2%
  WC 2014: 4.578 pts  exact=14.1%  correct=59.4%
  WC 2018: 4.391 pts  exact=17.2%  correct=56.2%
  WC 2022: 3.719 pts  exact=6.2%  correct=54.7%

CI lower > autofill: False
CI lower > GLM:      False


In [7]:
# ── Convergence diagnostics ───────────────────────────────────────────────
print('Convergence diagnostics per fold:')
diag_df = pd.DataFrame(fold_diags)
display(diag_df[['year', 'max_rhat', 'divergences', 'intercept_mean', 'home_adv_mean',
                  'sigma_attack_mean', 'sigma_defence_mean']])

Convergence diagnostics per fold:


,year,max_rhat,divergences,intercept_mean,home_adv_mean,sigma_attack_mean,sigma_defence_mean
0,2010,1.025445,8,0.026729,0.231731,0.126097,0.375931
1,2014,1.015392,0,0.098770,0.308608,0.243438,0.653408
2,2018,1.017032,0,0.111159,0.300857,0.277600,0.681355
3,2022,1.010175,0,0.105704,0.290059,0.320161,0.726944


In [8]:
# ── MLflow logging ───────────────────────────────────────────────────────
mlflow.set_tracking_uri('sqlite:////Users/seppe.vanswegenoven/projects/wk_2026_match_predictor/mlflow.db')
mlflow.set_experiment('wk2026-tabular-2026-06-13')

with mlflow.start_run(run_name='bayesian_hierarchical_poisson_loto'):
    mlflow.set_tags({
        'modality': 'tabular',
        'approach': 'bayesian_hierarchical_poisson',
        'eval_strategy': 'loto_cv',
        'dataset_version': 'split_v2',
        'model_framework': 'pymc',
        'literature': 'Baio_Blangiardo_2010',
    })
    mlflow.log_params({
        'draws': 1000,
        'tune': 1000,
        'chains': 2,
        'target_accept': 0.9,
        'score_strategy': 'max_expected_sporza_pts',
        'max_goals_grid': MAX_GOALS,
        'elo_prior_scale': 0.3,
        'sigma_attack_prior': 'HalfNormal(0.3)',
        'sigma_defence_prior': 'HalfNormal(0.3)',
        'home_adv_prior': 'Normal(0.2, 0.15)',
    })
    mlflow.log_metric('loto_mean_sporza_pts', ci['mean'])
    mlflow.log_metric('loto_ci_lo', ci['ci_lo'])
    mlflow.log_metric('loto_ci_hi', ci['ci_hi'])
    mlflow.log_metric('autofill_baseline', autofill_mean)
    mlflow.log_metric('delta_vs_autofill', ci['mean'] - autofill_mean)
    mlflow.log_metric('delta_vs_glm', ci['mean'] - 4.336)
    for r in fold_results:
        mlflow.log_metric(f'fold_{r["year"]}_mean_pts', r['mean_pts'])
        mlflow.log_metric(f'fold_{r["year"]}_pct_exact', r['pct_exact'])
    for d in fold_diags:
        mlflow.log_metric(f'fold_{d["year"]}_max_rhat', d['max_rhat'])
        mlflow.log_metric(f'fold_{d["year"]}_divergences', d['divergences'])
    run_id = mlflow.active_run().info.run_id

print(f'MLflow run_id: {run_id}')

MLflow run_id: a8873b42395242118040d7af708d3353


## Interpretation

The key questions to answer from the results above:

1. **Did hierarchical shrinkage fix the overfitting?** Compare WC 2010 fold (219 training rows, most sparse) to DC full MLE (2.922 pts on WC 2022 / 3.762 pooled).
2. **Did it beat the Poisson GLM?** GLM = 4.336 [3.930, 4.742]. The Bayesian model should improve if the per-team residuals (beyond ELO/form features) carry signal.
3. **Convergence**: R̂ < 1.05 and zero divergences are healthy. If divergences > 0, the prior may need reparameterisation (non-centred).
4. **Home advantage posterior**: Should be positive (~0.2–0.3 on log scale); near-zero would suggest neutral-venue correction is absorbing it.